In [151]:
import numpy as np
import PACKAGE_LAB

import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib import colors as mcolors
from importlib import reload
PACKAGE_LAB = reload(PACKAGE_LAB)

from PACKAGE_LAB import 

from package_DBR import SelectPath_RT, Delay_RT, Process




In [ ]:
#!!!!  ATTENTION Télécharger dash

import dash
from dash import dcc, html
import plotly.graph_objects as go
from dash.dependencies import Input, Output


In [152]:
help(LL_RT)

Help on function LL_RT in module PACKAGE_LAB:

LL_RT(MV, Kp, TLead, TLag, Ts, PV, PVInit=0, method='EBD')



In [153]:



# Définition des paramètres de simulation
TSim = 100
Ts = 0.5
N = int(TSim / Ts) + 1
theta = 5

# Définition du chemin MV
MVPath = {0: 0, 5: 1, 50: 2, 80: 3, TSim: 3}

# Initialisation des listes
t = []
MV = []
MVDelay = []
PV_EBD_1 = []
PV_EFD_1 = []

# Génération des données MV et MV_Delay
for i in range(0, N):
    t.append(i * Ts)
    SelectPath_RT(MVPath, t, MV)
    Delay_RT(MV, theta, Ts, MVDelay)

# Initialisation de l'application Dash
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Simulation du système Lead-Lag"),
    
    dcc.Graph(id='mv-graph'),
    dcc.Graph(id='pv-graph'),
    
    html.Label("TLead"),
    dcc.Slider(id='TLead', min=-10, max=50, step=1, value=-5, marks={i: str(i) for i in range(-10, 51, 10)}),
    
    html.Label("TLag"),
    dcc.Slider(id='TLag', min=0, max=50, step=1, value=10, marks={i: str(i) for i in range(0, 51, 10)}),
    
    html.Label("Kp"),
    dcc.Slider(id='Kp', min=0, max=10, step=0.1, value=1, marks={i: str(i) for i in range(0, 11, 2)})
])

@app.callback(
    [Output('mv-graph', 'figure'), Output('pv-graph', 'figure')],
    [Input('TLead', 'value'),
     Input('TLag', 'value'),
     Input('Kp', 'value')]
)
def update_graph(TLead, TLag, Kp):
    PV_EBD_1 = []
    PV_EFD_1 = []
    
    for i in range(0, N):
        LL_RT(MVDelay[0:i+1], Kp, TLead, TLag, Ts, PV_EBD_1)
        LL_RT(MVDelay[0:i+1], Kp, TLead, TLag, Ts, PV_EFD_1, method='EFD')
    
    # Graphique MV
    mv_fig = go.Figure()
    mv_fig.add_trace(go.Scatter(x=t, y=MVDelay, mode='lines', name='MV Delayed'))
    mv_fig.add_trace(go.Scatter(x=t, y=MV, mode='lines', name='MV'))
    mv_fig.update_layout(
        title="MV Graph",
        xaxis_title="Time (s)",
        yaxis_title="MV [°C]",
        height=400,
        template="plotly_white"
    )
    
    # Graphique PV
    pv_fig = go.Figure()
    pv_fig.add_trace(go.Scatter(x=t, y=PV_EBD_1, mode='lines', name='PV with EBD'))
    pv_fig.add_trace(go.Scatter(x=t, y=PV_EFD_1, mode='lines', name='PV with EFD'))
    pv_fig.update_layout(
        title="PV Graph",
        xaxis_title="Time (s)",
        yaxis_title="PV [°C]",
        height=400,
        template="plotly_white"
    )
    
    return mv_fig, pv_fig

if __name__ == '__main__':
    app.run_server(debug=True)
